# **CENG 786 - Robot Motion Planning and Control- Fall 2024**
## Homework 2 - Potential Fields

### Instructor: Uluç Saranlı
### Student: Çağdaş Güven - 2738938

---

### **Abstract**
In this project, I explored the implementation and evaluation of motion planning algorithms for a planar point robot navigating a 2D sphere world with circular obstacles and boundaries. Two approaches, Potential Fields and Navigation Functions, were employed to guide the robot from a start position to a goal while avoiding obstacles. The Potential Field method combined attractive and repulsive forces, leveraging gradients to direct the robot, while the Navigation Function ensured a global minimum at the goal for smooth navigation. Comparative analyses highlighted their performance, robustness, and effectiveness across various obstacle configurations. Results demonstrated the strengths and limitations of both methods, providing insights into their applicability in robotic motion planning tasks.

---

## **1. Introduction**
Motion planning is a fundamental aspect of robotics, enabling autonomous navigation in environments with obstacles. In this project, I focused on comparing two widely recognized motion planning techniques: Potential Fields and Navigation Functions. The Potential Field method utilized attractive forces to pull the robot towards the goal and repulsive forces to avoid obstacles, relying on gradient descent for path generation. Conversely, the Navigation Function employed a globally optimal potential field that guaranteed a single minimum at the goal, addressing common issues like local minima.

The sphere world environment, characterized by circular obstacles and boundaries, served as the domain for this investigation. Configurable parameters such as obstacle locations and sizes enabled a thorough evaluation of algorithm performance under varying complexities. The project provided an in-depth understanding of the strengths and weaknesses of these methods, emphasizing their path efficiency, computational demands, and robustness in challenging scenarios.

---

## **2. Literature Review**

Motion planning is a critical aspect of robotic autonomy, enabling robots to navigate toward a goal while avoiding obstacles. This project focuses on implementing and comparing Artificial Potential Fields (APFs) and Navigation Functions (NFs) in a 2D sphere world. While both methods are grounded in potential field-based navigation, they differ in robustness, computational requirements, and handling of environmental complexities. This section synthesizes insights from relevant research, connecting them to the design and implementation of the project.

#### **2.1 Artificial Potential Fields (APFs)**
APFs are a reactive planning approach that combines attractive forces toward the goal and repulsive forces to avoid obstacles. These forces are derived from potential functions that mimic physical phenomena like gravity and electrostatic interactions. Khatib’s seminal work laid the foundation for this methodology【4】.

Despite their computational efficiency and simplicity, APFs have notable limitations:
1. **Local Minima Problem**: Robots can get stuck at points where the attractive and repulsive forces cancel out【3】【4】.  
2. **Oscillations**: In dense environments, robots may exhibit oscillatory behaviors when navigating narrow passages【2】【3】.  
3. **Goal Non-Reachable with Obstacles Nearby (GNRON)**: If the goal is close to obstacles, the robot may fail to converge【3】【4】.  

In my project, these challenges were observed during the implementation of APFs in scenarios with multiple obstacles. Enhancements in literature aim to mitigate these issues:
- **Dynamic Repulsive Potential Factor (DRPF)**: Modifies repulsive potentials dynamically to improve obstacle avoidance in dense environments【3】.  
- **Deterministic Annealing APFs (DA-APF)**: Incorporates a temperature parameter to overcome local minima, similar to simulated annealing in optimization【3】.  
- **Reinforced Potential Fields (RPF)**: Combines APFs with reinforcement learning to improve path smoothness and avoid oscillations, particularly in dynamic environments【2】.  

The inclusion of tangential forces in my implementation aligns with the approaches proposed to enhance trajectory smoothness and obstacle avoidance.

#### **2.2 Navigation Functions (NFs)**
Navigation Functions extend the concept of APFs by ensuring a single global minimum at the goal. This property guarantees convergence and eliminates the local minima issue, making NFs more robust in complex environments【2】【4】.

However, the robustness of NFs comes at a computational cost. In my project, the mathematical formulation and computational intensity of NFs were evident, especially in environments with numerous obstacles. The literature supports the use of NFs in environments requiring high reliability but with allowances for greater computational resources:
- Zhang et al. introduced a Navigation Function with augmented sensing to improve adaptability in dynamic and cluttered environments【2】.  
- Pioneering work on NFs emphasized their applicability in handling real-world robotic constraints like non-holonomic motion【4】【7】.  

#### **2.3 Comparative Analysis**
This project compares APFs and NFs based on their performance in a 2D sphere world. The literature underscores the following key trade-offs:
- **Efficiency vs. Robustness**: APFs are computationally efficient but less reliable in complex scenarios. NFs, while computationally demanding, provide guaranteed convergence to the goal【3】【4】【5】.  
- **Flexibility**: APFs adapt better to dynamic obstacles with modifications like DRPF, while NFs are more suitable for static, well-modeled environments【2】【5】.

---

## **3. Methodology**

The project involves implementing and analyzing two motion planning algorithms—Potential Fields and Navigation Functions—within a simulated 2D sphere world. The sphere world is characterized by circular obstacles, a circular boundary, and defined start and goal positions.

---

### **1. Potential Fields**

- **Overview**:  
  The Potential Field method creates a virtual field by combining attractive forces to guide the robot toward the goal and repulsive forces to push the robot away from obstacles. The robot moves in the direction of the **negative gradient** of this field to ensure navigation toward the goal while avoiding collisions.

#### **Formulas**
1. **Attractive Potential:**
   $\displaystyle U_{attr}(q) = \frac{1}{2} \epsilon_q \|q - q_{goal}\|^2$  
   The corresponding attractive force is:  
   $\displaystyle F_{attr}(q) = -\nabla U_{attr}(q) = -\epsilon_q (q - q_{goal})$

2. **Repulsive Potential:**
   $\displaystyle U_{rep}(q) = 
   \begin{cases} 
   \frac{1}{2} \epsilon_r \left(\frac{1}{d(q, q_{obs})} - \frac{1}{Q_{inf}}\right)^2 & \text{if } d(q, q_{obs}) \leq Q_{inf}, \\
   0 & \text{if } d(q, q_{obs}) > Q_{inf},
   \end{cases}$  
   where $d(q, q_{obs})$ is the distance to the obstacle and $Q_{inf}$ is the influence radius.  
   The corresponding repulsive force is:  
   $\displaystyle F_{rep}(q) = -\nabla U_{rep}(q)$

3. **Total Gradient:**
   $\displaystyle F_{total}(q) = F_{attr}(q) + \sum_{i} F_{rep}^i(q)$

#### **Implementation**
The robot's position is updated iteratively using the gradient descent formula:  
$\displaystyle q_{new} = q_{current} - \lambda \frac{F_{total}(q_{current})}{\|F_{total}(q_{current})\|}$  
where $\lambda$ is the step size, ensuring controlled motion.

---

### **2. Navigation Functions**

- **Overview**:  
  Navigation Functions are designed to provide a **globally optimal potential field**. Unlike standard APFs, they eliminate local minima by ensuring that the goal is the sole minimum in the field.

#### **Formulas**
1. **Navigation Function (Potential Field):**
   $\displaystyle U_{nav}(q) = \frac{\gamma(q)^\kappa}{\gamma(q)^\kappa + \beta(q)}$  
   - $\gamma(q) = \|q - q_{goal}\|^2$ is the term that pulls the robot toward the goal.  
   - $\beta(q)$ represents the obstacle term:  
     $\displaystyle \beta(q) = \beta_0(q) \prod_{i=1}^n \beta_i(q)$,  
     where $\beta_0(q)$ accounts for the world boundary, and $\beta_i(q)$ represents individual obstacles.

2. **Gradient of the Navigation Function:**
   $\displaystyle \nabla U_{nav}(q) = \frac{\kappa \gamma(q)^{\kappa-1} \nabla \gamma(q) \beta(q) - \gamma(q)^\kappa \nabla \beta(q)}{\left(\gamma(q)^\kappa + \beta(q)\right)^2}$

#### **Implementation**
The robot follows the negative gradient of $U_{nav}(q)$ to reach the goal. The path is computed using MATLAB's `ode45` solver to numerically integrate the motion.

---

### **3. Experimental Setup**

- **Environment Configuration**:  
  - A 2D circular boundary is defined as the world, with configurable obstacle locations and radii.
  - The start and goal positions are predefined within the boundary.

- **Evaluation Metrics**:  
  The algorithms are evaluated based on:
  1. **Path Smoothness**: The continuity of the trajectory.
  2. **Time to Convergence**: Time taken to reach the goal.
  3. **Collision Avoidance**: Success in avoiding obstacles.
  4. **Goal Reaching**: The robot’s ability to reach the goal.

- **Visualization**:  
  - Plots of the generated trajectories and potential fields are created in MATLAB.
  - Distance to the goal over time is plotted to illustrate convergence.

---

### **4. Main Algorithm Workflow**

#### **Initialization**
1. Set world parameters: Define boundary radius and center.
2. Specify start and goal positions.
3. Define obstacles with their positions and radii.

#### **Potential Field Algorithm**
1. Compute **Attractive Force**:  
   $\displaystyle F_{attr}(q) = -\epsilon_q (q - q_{goal})$
2. Compute **Repulsive Forces** for each obstacle:  
   - If $d(q, q_{obs}) < Q_{inf}$:  
     $\displaystyle F_{rep}(q) = \epsilon_r \left(\frac{1}{d(q, q_{obs})} - \frac{1}{Q_{inf}}\right) \frac{1}{d(q, q_{obs})^2} (q - q_{obs})$
   - Add a tangential force component for smoother navigation:  
     $\displaystyle F_{tangential} = k_t \cdot \frac{[-\Delta y; \Delta x]}{\|q - q_{obs}\|}$
   - Combine repulsive and tangential forces.

3. Calculate the **Total Gradient**:  
   $\displaystyle F_{total}(q) = F_{attr}(q) + \sum_{i} F_{rep}^i(q)$

4. Update the robot’s position iteratively:  
   $\displaystyle q_{new} = q_{current} - \lambda \frac{F_{total}(q_{current})}{\|F_{total}(q_{current})\|}$

#### **Navigation Function Algorithm**
1. Compute the gradient of the navigation function:  
   $\displaystyle \nabla U_{nav}(q) = \frac{\kappa \gamma(q)^{\kappa-1} \nabla \gamma(q) \beta(q) - \gamma(q)^\kappa \nabla \beta(q)}{\left(\gamma(q)^\kappa + \beta(q)\right)^2}$
2. Update the robot’s position using the negative gradient.

#### **Visualization**
1. Plot the world boundary and obstacles.
2. Overlay the robot’s generated trajectory.
3. Plot distance to the goal vs. time.


## **4. Results and Discussion**

![single_obstacle_reference](images/single_obstacle_reference.png)
image 1 is the reference trajectory for the single obstacle
![single_obstacle_strong_push](images/single_obstacle_strong_push.png)
image 2 is the trajectory for the single obstacle with strong repulsive force
![single_obstacle_stronger_pull](images/single_obstacle_stronger_pull.png)
image 3 is the trajectory for the single obstacle with stronger pull force

As observed in the images, Image 1 illustrates the reference trajectory where the robot follows a relatively smooth path to the goal by adhering to the gradient of the potential field. This trajectory demonstrates balanced interaction between the attractive and repulsive forces, allowing the robot to efficiently navigate the environment.

In Image 2, the robot is subjected to a stronger repulsive force compared to the reference trajectory, causing it to be pushed significantly away from the obstacle. Notably, despite the absence of border repulsion, the robot is diverted away from the goal toward the center of the world. This behavior suggests that when the repulsive force outweighs the attractive force in my implementation, the algorithm struggles to find a feasible path to the goal. I attribute this issue to the calculation of the tangential multiplier in the repulsive force computation. While heuristic-based distance checks could potentially mitigate this issue, I chose not to implement them as such an approach, although effective in a spherical world, might not generalize well and could be conceptually flawed.

In Image 3, the robot follows a trajectory that more closely resembles a heuristic path. While the obstacle avoidance distance is reduced compared to the reference trajectory, the robot is still pushed away from the goal. This behavior is likely due to the attractive force being significantly stronger than the repulsive force, resulting in a more realistic trajectory that balances obstacle avoidance with goal-directed motion.

![2_obstacles_reference](images/2_obstacles_reference.png)
image 4 is the reference trajectory for the two obstacles
![2_obstacle_strong_push](images/2_obstacle_strong_push.png)
image 5 is the trajectory for the two obstacles with strong repulsive force
![2_obstacles_strong_pull](images/2_obstacles_strong_pull.png)
image 6 is the trajectory for the two obstacles with stronger pull force

As depicted in Image 4, the path terminates at a local minimum, which appears to result from the comparable magnitudes of the attractive and repulsive forces. In Image 5, increasing the repulsive force enables the robot to successfully avoid obstacles and reach the goal. Conversely, in Image 6, the attractive force is significantly stronger than the repulsive force, compelling the robot to navigate between obstacles. This behavior also highlights the long-range influence of obstacles on the robot's trajectory, which could be fine-tuned in future work.

When considering scenarios with one or two obstacles, potential field applications demand meticulous tuning of parameters, including the repulsive force, attractive force, influence radius, critical distance, and the relative spacing between obstacles. To mitigate local minima issues, implementing a navigation switch mechanism could be beneficial. This approach would employ heuristic-based distance checks to make decisions based on the robot's spatial context, ensuring smoother navigation and better obstacle avoidance.

## **5. Conclusion**

In this project, I implemented and evaluated two motion planning algorithms, Potential Fields and Navigation Functions, within a 2D spherical world to navigate a robot from a start position to a goal while avoiding obstacles. The analysis revealed the strengths and limitations of each method, emphasizing the importance of carefully tuning key parameters, such as the balance between attractive and repulsive forces, influence radius, and critical distances, to achieve successful navigation.

The simulations demonstrated that the Potential Field method effectively guides the robot in obstacle-avoidance scenarios but is susceptible to local minima when the attractive and repulsive forces are of similar magnitudes. The Navigation Function method, while computationally more demanding, ensures global convergence to the goal by design, avoiding local minima entirely.

An important observation made during the project is the relationship between simulation time and the step size parameter λλ. While larger λλ values reduce simulation time by allowing the robot to make larger steps, they can potentially compromise path smoothness and increase the likelihood of collisions. On the other hand, using smaller λλ values provides finer control over the robot’s movement but significantly increases computation time, especially in scenarios with dense obstacle configurations. I observed that simulation time has a greater impact on overall computation time than the step size alone.

To address these issues, a potential improvement would be to implement a dynamic λλ calculation that adapts to the robot’s distance from the goal and obstacles. This approach could optimize the trade-off between path smoothness and computational efficiency, making the system more versatile and scalable to complex environments.

Overall, the findings of this project provide insights into the practical challenges and design considerations of motion planning algorithms, highlighting avenues for further optimization and application in real-world robotic systems.


## **References**

1. Aziz ur Rehman, Ahsan Tanveer, et al., "Motion Planning for Autonomous Ground Vehicles Using Artificial Potential Fields: A Review," 2023, DOI:10.22581/muet1982.220X.  
2. Dengyu Zhang, Xinyu Zhang, et al., "Reinforced Potential Field for Multi-Robot Motion Planning in Cluttered Environments," 2023, DOI:2307.14110.  
3. Zhengtian Wu, Jinyu Dai, et al., "Robot Path Planning Based on Artificial Potential Field with Deterministic Annealing," ISA Transactions, 2023, DOI:10.1016/j.isatra.2023.02.018.  
4. O. Khatib, "Real-Time Obstacle Avoidance for Manipulators and Mobile Robots," International Journal of Robotics Research, 1986.  
5. J. Borenstein, Y. Koren, "The Vector Field Histogram—Fast Obstacle Avoidance for Mobile Robots," IEEE Transactions on Robotics and Automation, 1991.  
6. T. Lozano-Perez, "Spatial Planning: A Configuration Space Approach," IEEE Transactions on Computers, 1983.  
7. J. Minguez, L. Montano, "Nearness Diagram Navigation (ND): Collision Avoidance in Troublesome Scenarios," IEEE Transactions on Robotics and Automation, 2004.  
8. P. R. S. Barbosa et al., "Dynamic Window Approach for Local Path Planning," IEEE Robotics and Automation Letters, 2021.  
9. C. Zhou et al., "Dynamic Artificial Potential Fields in Mobile Robot Motion Planning," Applied Soft Computing, 2020.  
10. J. H. Reif, "Complexity of the Mover’s Problem and Generalizations," Annual ACM Symposium on Theory of Computing, 1979.